# Practical 1 — Sentinel-2 access and pre-processing with CDSE

**Study area:** Adventdalen, Svalbard  
**Environment:** Google Colab  
**Data:** Sentinel-2 Level-2A  
**Source:** Copernicus Data Space Ecosystem (CDSE)

**Workflow:** Copernicus Browser → STAC catalogue → selected acquisition → Process API → SCL quality mask → NDVI → GeoTIFF

Google Colab is the Python environment. CDSE provides the catalogue and satellite data.

## Before the practical

You should already have access to Google Colab, a CDSE account, and a CDSE OAuth Client ID and Client Secret.

Keep the Client Secret private. It will be entered interactively and will not be written into the notebook.

# Part A — Inspect the study area in Copernicus Browser

Open https://browser.dataspace.copernicus.eu/

1. Sign in.
2. Search for **Longyearbyen, Svalbard**.
3. Pan east to **Adventdalen**.
4. Open **SEARCH**.
5. Select **Sentinel-2 → MSI → L2A**.
6. Use **1 July–31 August 2025**.
7. Inspect two or three acquisitions.

The product-level cloud-cover value is useful for screening, but it does not describe cloud conditions specifically within our small AOI.

# Part B — Start the Colab workflow

## Step 1 — Install the additional packages

In [ ]:
!pip -q install requests-oauthlib rasterio pyproj folium

## Step 2 — Import the libraries

In [ ]:
from getpass import getpass
from io import BytesIO
import math

import folium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import rasterio

from IPython.display import HTML
from oauthlib.oauth2 import BackendApplicationClient
from PIL import Image
from pyproj import Transformer
from rasterio.io import MemoryFile
from requests_oauthlib import OAuth2Session

# Part C — Define the study area

## Step 3 — Define the Adventdalen AOI

The STAC catalogue uses geographic coordinates in **EPSG:4326**.  
The bounding box is given as `west, south, east, north`.

In [ ]:
AOI_WGS84 = (15.75, 78.16, 16.05, 78.24)

SEARCH_START = "2025-07-01"
SEARCH_END = "2025-08-31"

MAX_CLOUD_COVER = 50

## Step 3a — View the AOI on a Svalbard map

The rectangle shows the same bounding box. Use the layer control to switch between the Norwegian Polar Institute topographic map and its Copernicus Sentinel mosaic.

In [ ]:
west, south, east, north = AOI_WGS84
centre_lat = (south + north) / 2
centre_lon = (west + east) / 2

aoi_map = folium.Map(
    location=[centre_lat, centre_lon],
    zoom_start=10,
    tiles=None,
)

folium.TileLayer(
    tiles=("https://geodata.npolar.no/arcgis/rest/services/"
           "Basisdata/NP_Basiskart_Svalbard_WMTS_3857/"
           "MapServer/tile/{z}/{y}/{x}"),
    attr="Norwegian Polar Institute — Svalbard base map (CC BY 4.0)",
    name="NPI Svalbard topographic map",
    overlay=False,
    control=True,
    show=True,
).add_to(aoi_map)

folium.TileLayer(
    tiles=("https://geodata.npolar.no/arcgis/rest/services/"
           "Basisdata/NP_Satellitt_Svalbard_WMTS_3857/"
           "MapServer/tile/{z}/{y}/{x}"),
    attr="Norwegian Polar Institute; Copernicus Sentinel data — CC BY 4.0",
    name="NPI Copernicus Sentinel mosaic",
    overlay=False,
    control=True,
    show=False,
).add_to(aoi_map)

folium.Rectangle(
    bounds=[[south, west], [north, east]],
    tooltip="Adventdalen AOI",
    fill=True,
    fill_opacity=0.10,
    weight=3,
).add_to(aoi_map)

aoi_map.fit_bounds([[south, west], [north, east]])
folium.LayerControl(collapsed=False).add_to(aoi_map)
aoi_map

# Part D — Search Sentinel-2 Level-2A with STAC

## Step 4 — Build the catalogue query

STAC is used for discovery. No spectral raster is requested at this stage.

In [ ]:
STAC_SEARCH_URL = "https://stac.dataspace.copernicus.eu/v1/search"

stac_query = {
    "collections": ["sentinel-2-l2a"],
    "bbox": list(AOI_WGS84),
    "datetime": f"{SEARCH_START}T00:00:00Z/{SEARCH_END}T23:59:59Z",
    "query": {"eo:cloud_cover": {"lte": MAX_CLOUD_COVER}},
    "sortby": [{"field": "properties.eo:cloud_cover", "direction": "asc"}],
    "limit": 20,
}

## Step 5 — Send the STAC request

In [ ]:
stac_response = requests.post(
    STAC_SEARCH_URL,
    json=stac_query,
    timeout=60,
)
stac_response.raise_for_status()

items = stac_response.json()["features"]

if not items:
    raise RuntimeError("No Sentinel-2 L2A acquisitions matched the search criteria.")

print(f"Acquisitions returned: {len(items)}")

## Step 6 — Create the scene table

The table keeps only the information needed to choose an acquisition.

In [ ]:
scenes = pd.DataFrame(
    {
        "datetime": item["properties"]["datetime"],
        "cloud_cover": item["properties"].get("eo:cloud_cover", np.nan),
        "product_id": item["id"],
    }
    for item in items
)

scenes["datetime"] = pd.to_datetime(scenes["datetime"], utc=True)

scenes = (
    scenes
    .sort_values(["cloud_cover", "datetime"])
    .reset_index(drop=True)
)

scenes.index.name = "scene_index"

table = scenes[["datetime", "cloud_cover", "product_id"]]

HTML(
    table.to_html(
        index=True,
        float_format=lambda value: f"{value:.1f}",
    )
)

## Step 7 — Select one acquisition

Choose the scene index after comparing the table with the scenes inspected in Copernicus Browser.

In [ ]:
SCENE_INDEX = 0

selected = scenes.loc[SCENE_INDEX]
display(selected)

## Step 7a — Preview the selected product

The STAC Item contains a quicklook of the complete Sentinel-2 product. This is a final visual check of the selected acquisition, not a crop of the Adventdalen AOI.

In [ ]:
selected_product_id = selected["product_id"]

ITEM_URL = (
    "https://stac.dataspace.copernicus.eu/v1/"
    f"collections/sentinel-2-l2a/items/{selected_product_id}"
)

item_response = requests.get(ITEM_URL, timeout=60)
item_response.raise_for_status()

selected_item = item_response.json()
thumbnail_url = selected_item["assets"]["thumbnail"]["href"]

In [ ]:
thumbnail_response = requests.get(thumbnail_url, timeout=60)
thumbnail_response.raise_for_status()

quicklook = Image.open(BytesIO(thumbnail_response.content))

plt.figure(figsize=(8, 8))
plt.imshow(quicklook)
plt.title(
    f"Selected Sentinel-2 L2A product\n"
    f"{selected['datetime'].date()} | "
    f"cloud cover: {selected['cloud_cover']:.1f}%"
)
plt.axis("off")
plt.show()

# Part E — Prepare access to the selected observation

## Step 8 — Use the date of the selected acquisition

The STAC catalogue has already identified the observation. For the Process API we use the **same calendar day**, rather than an artificial minute-scale time window.

In [ ]:
selected_time = pd.Timestamp(selected["datetime"])
selected_date = selected_time.strftime("%Y-%m-%d")

time_from = f"{selected_date}T00:00:00Z"
time_to = f"{selected_date}T23:59:59Z"

print("Selected acquisition:", selected_time)
print("Process API date:", selected_date)

## Before continuing — create a CDSE OAuth client

The STAC catalogue can be searched without authentication, but the CDSE processing service requires an authenticated request.

1. Open https://shapps.dataspace.copernicus.eu/dashboard/
2. Sign in with your CDSE account.
3. Open **User Settings** → **OAuth clients**.
4. Click **Create** and use a simple name, for example `RSTC2026-Colab`.
5. Copy the **Client ID** and **Client Secret**.

The Client Secret is displayed only once. Do not save it in the notebook or GitHub.

## Step 9 — Create an OAuth session

The credentials are used to obtain a temporary access token for the processing service.

In [ ]:
CLIENT_ID = input("CDSE OAuth Client ID: ").strip()
CLIENT_SECRET = getpass("CDSE OAuth Client Secret: ")

TOKEN_URL = (
    "https://identity.dataspace.copernicus.eu/"
    "auth/realms/CDSE/protocol/openid-connect/token"
)

client = BackendApplicationClient(client_id=CLIENT_ID)
oauth = OAuth2Session(client=client)

oauth.fetch_token(
    token_url=TOKEN_URL,
    client_secret=CLIENT_SECRET,
    include_client_id=True,
)

print("Authenticated session created.")

# Part F — Prepare the raster grid

## Step 10 — Transform the AOI to UTM zone 33N

The STAC search used longitude and latitude. For raster processing we use **EPSG:32633**, so the output resolution is expressed directly in metres.

In [ ]:
transformer = Transformer.from_crs(
    "EPSG:4326",
    "EPSG:32633",
    always_xy=True,
)

left, bottom, right, top = transformer.transform_bounds(
    *AOI_WGS84,
    densify_pts=21,
)

AOI_UTM = (left, bottom, right, top)
AOI_UTM

## Step 11 — Check the output size

B02, B03, B04 and B08 have a native resolution of 10 m.

In [ ]:
RESOLUTION = 10

width = math.ceil((AOI_UTM[2] - AOI_UTM[0]) / RESOLUTION)
height = math.ceil((AOI_UTM[3] - AOI_UTM[1]) / RESOLUTION)

print(f"Expected output size: {width} × {height} pixels")

# Part G — Request cloud- and shadow-screened Sentinel-2 data

## Step 12 — Define a simple quality screen

For this introductory workflow, the Process API performs a basic quality screen before the raster is returned to Colab.

We use:

- **`dataMask`** to exclude pixels where source data are not available;
- **SCL** to exclude saturated/defective pixels, cast and cloud shadows, unclassified pixels, medium/high-probability cloud and thin cirrus.

We deliberately retain **snow/ice (SCL 11)** because snow is examined in the next course block.

This is a practical first-pass mask for a short exercise. More demanding time-series analyses may require a more detailed cloud-quality strategy.

## Step 13 — Define the bands and apply the mask server-side

The server returns:

1. B02 — blue;
2. B03 — green;
3. B04 — red;
4. B08 — near infrared;
5. a binary valid-pixel mask.

Invalid pixels are set to zero on the server. In the following step, the binary mask is used to convert those zeros to `NaN` before analysis.

In [ ]:
evalscript = r"""
//VERSION=3

function setup() {
    return {
        input: [{
            bands: [
                "B02", "B03", "B04", "B08",
                "SCL", "dataMask"
            ],
            units: [
                "REFLECTANCE", "REFLECTANCE",
                "REFLECTANCE", "REFLECTANCE",
                "DN", "DN"
            ]
        }],
        output: {
            bands: 5,
            sampleType: "FLOAT32"
        }
    };
}

function isValidPixel(sample) {
    if (sample.dataMask === 0) {
        return false;
    }

    const invalidSCL = [0, 1, 2, 3, 7, 8, 9, 10];

    return !invalidSCL.includes(sample.SCL);
}

function evaluatePixel(sample) {
    const valid = isValidPixel(sample) ? 1.0 : 0.0;

    return [
        sample.B02 * valid,
        sample.B03 * valid,
        sample.B04 * valid,
        sample.B08 * valid,
        valid
    ];
}
"""

## Step 14 — Build the Process API request

The request uses:

- Sentinel-2 Level-2A;
- the date selected through STAC;
- the Adventdalen AOI;
- a 10 m output grid.

SCL has a native resolution of 20 m, so nearest-neighbour upsampling is used for the categorical quality information.

In [ ]:
PROCESS_URL = "https://sh.dataspace.copernicus.eu/process/v1"

process_request = {
    "input": {
        "bounds": {
            "bbox": list(AOI_UTM),
            "properties": {
                "crs": "http://www.opengis.net/def/crs/EPSG/0/32633"
            },
        },
        "data": [{
            "type": "sentinel-2-l2a",
            "dataFilter": {
                "timeRange": {
                    "from": time_from,
                    "to": time_to,
                },
                "mosaickingOrder": "leastCC",
                "maxCloudCoverage": MAX_CLOUD_COVER,
            },
            "processing": {
                "upsampling": "NEAREST",
            },
        }],
    },
    "output": {
        "resx": RESOLUTION,
        "resy": RESOLUTION,
        "responses": [{
            "identifier": "default",
            "format": {
                "type": "image/tiff"
            },
        }],
    },
    "evalscript": evalscript,
}

## Step 15 — Request the raster

The spatial subset, band selection and basic quality screen are performed on the server. Only the requested five-layer raster is returned to Colab.

In [ ]:
process_response = oauth.post(
    PROCESS_URL,
    json=process_request,
    headers={"Accept": "image/tiff"},
    timeout=120,
)

process_response.raise_for_status()

print("Status:", process_response.status_code)
print(
    "Content type:",
    process_response.headers.get("content-type"),
)
print(
    "Bytes received:",
    len(process_response.content),
)

# Part H — Read and check the screened data

## Step 16 — Read the GeoTIFF directly from memory

Nothing is written to disk. The API response is opened directly from memory.

In [ ]:
with MemoryFile(process_response.content) as memfile:
    with memfile.open() as src:
        stack = src.read()
        raster_crs = src.crs
        raster_bounds = src.bounds

print("Array shape:", stack.shape)
print("CRS:", raster_crs)
print("Bounds:", raster_bounds)

## Step 17 — Separate the spectral bands and valid-pixel mask

The server returned zeros for screened pixels. The binary mask is used here to represent those pixels as `NaN`, so they are excluded naturally from subsequent calculations.

In [ ]:
blue = stack[0].astype(np.float32)
green = stack[1].astype(np.float32)
red = stack[2].astype(np.float32)
nir = stack[3].astype(np.float32)

valid_mask = stack[4] > 0.5

for band in (blue, green, red, nir):
    band[~valid_mask] = np.nan

valid_fraction = 100 * valid_mask.mean()

print(
    f"Pixels retained after basic quality screening: "
    f"{valid_fraction:.1f}%"
)

## Step 18 — Check the reflectance ranges

The retained pixels should contain non-zero surface-reflectance values. This is a quick check before visualisation.

In [ ]:
for name, band in {
    "B02 — Blue": blue,
    "B03 — Green": green,
    "B04 — Red": red,
    "B08 — Near infrared": nir,
}.items():

    finite = band[np.isfinite(band)]

    print(
        f"{name}: "
        f"min = {finite.min():.4f}, "
        f"max = {finite.max():.4f}"
    )

# Part I — Display a vegetation-sensitive colour composite

## Step 19 — Create a colour-infrared composite

A **NIR–Red–Green** composite places:

- B08 (near infrared) in the red display channel;
- B04 (red) in the green display channel;
- B03 (green) in the blue display channel.

Healthy vegetation typically appears in red tones because vegetation reflects strongly in the near infrared.

The stretch below is used only for display. The original reflectance arrays remain unchanged.

In [ ]:
cir = np.dstack(
    [nir, red, green]
)

cir_display = np.zeros_like(
    cir,
    dtype=np.float32,
)

for band_index in range(3):
    values = cir[..., band_index]
    finite = values[np.isfinite(values)]

    low, high = np.percentile(
        finite,
        [2, 98],
    )

    stretched = (
        (values - low) / (high - low)
    )

    cir_display[..., band_index] = np.clip(
        stretched,
        0,
        1,
    )

cir_display = np.nan_to_num(
    cir_display,
    nan=0.0,
)

In [ ]:
plt.figure(figsize=(9, 6))

plt.imshow(cir_display)

plt.title(
    f"Sentinel-2 L2A — NIR–Red–Green composite\n"
    f"Adventdalen — {selected_date}"
)

plt.axis("off")
plt.show()

# Part J — Calculate NDVI

## Step 20 — Calculate NDVI from the screened reflectance bands

\[
NDVI = \frac{B08 - B04}{B08 + B04}
\]

Because the quality mask has already been applied, screened pixels remain `NaN` and are ignored by the calculation.

In [ ]:
denominator = nir + red

ndvi = np.full(
    red.shape,
    np.nan,
    dtype=np.float32,
)

usable = (
    np.isfinite(red)
    & np.isfinite(nir)
    & (np.abs(denominator) > 1e-6)
)

ndvi[usable] = (
    (nir[usable] - red[usable])
    / denominator[usable]
)

## Step 21 — Display NDVI

Use the map as a rapid check of the processing chain rather than a detailed ecological interpretation.

In [ ]:
plt.figure(figsize=(9, 6))

image = plt.imshow(
    ndvi,
    vmin=-0.2,
    vmax=0.8,
    cmap="RdYlGn",
)

plt.colorbar(
    image,
    label="NDVI",
)

plt.title(
    f"NDVI — Adventdalen — {selected_date}"
)

plt.axis("off")
plt.show()

# End of practical

The final workflow is:

**Copernicus Browser**  
visual inspection

↓  

**CDSE STAC catalogue**  
find and select an acquisition

↓  

**CDSE Process API**  
select B02, B03, B04 and B08  
+ basic SCL/dataMask quality screening

↓  

**Google Colab**  
colour-infrared composite → NDVI

No raster files are downloaded or written to disk during the exercise. The data are requested, processed and visualised in memory.

The main distinction remains:

**discovery → selective access → quality screening → analysis**